# SQLAlchemy

In [1]:
import os                   # 운영체제 관련 표준 라이브러리
import pathlib              # 파일 경로를 객체로 다루는 표준 라이브러리

here = pathlib.Path.cwd()   # 현재 작업 위치 확인

ROOT=here.parents[2] if here.name=='0910Th' else here    # hanwha-agent

os.chdir(ROOT)  # 앞으로 모든 상대경로는 이 폴더가 기준이 된다.

In [3]:
SANDBOX=ROOT/'sandbox'/'w2'/'0910Th'

# SQLite: 파이썬 표준 라이브러리. 별도 설치 불필요.
import sqlite3

# 실습용 DB 파일 생성 경로
db_path=SANDBOX / 'sql_practice.db'

# DB파일 연결: 파일 없으면 신규 생성
conn=sqlite3.connect(db_path)

# 커서 cursor: DB에 명령을 보내고 결과를 받아오는 창구
cur=conn.cursor()

print('successfully connected:',db_path)

successfully connected: d:\hanwha-agent\sandbox\w2\0910Th\sql_practice.db


* SQLite 연습

In [4]:
# 테이블 생성
sql='''
CREATE TABLE documents(
    id INTEGER PRIMARY KEY,
    doc_id VARCHAR(20) NOT NULL,
    title VARCHAR(200) NOT NULL,
    version VARCHAR(10) NOT NULL,
    department VARCHAR(15) NOT NULL,
    security_level VARCHAR(10) NOT NULL,
    effective_date DATE NOT NULL,
    expiry_date DATE,
    is_latest BOOLEAN NOT NULL DEFAULT FALSE,
    page_count INTEGER NOT NULL DEFAULT 0,
    create_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);
'''
cur.execute(sql) # 쿼리문 실행

print('DB 테이블 생성')

# 어떤 테이블이 있는지 확인
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
print('Table List:',cur.fetchall())

DB 테이블 생성
Table List: [('documents',)]


In [6]:
# 데이터 저장
sql='''
INSERT INTO documents(doc_id, title, version, department, security_level, effective_date, expiry_date, is_latest, page_count)
VALUES(?,?,?,?,?,?,?,?,?);
'''
cur.execute(sql, ('DOC-HR-012','출장 여비 규정','1.0','인사팀','일반','2025-01-01','2026-12-31',False,28))
print('successfully saved')

successfully saved


In [9]:
# 데이터 조회
sql='SELECT id, doc_id, title, version, department FROM documents;'
cur.execute(sql,())

result=cur.fetchall()
print(result)

# for i,r in enumerate(result):
#     print(r[i])

# for r in result:
#     print([str(val) + '/' for i, val in enumerate(r)])

for r in result:
    id=r[0]
    doc_id=r[1]
    title=r[2]
    print(id, doc_id, title)

[(1, 'DOC-HR-012', '출장 여비 규정', '1.0', '인사팀')]
1 DOC-HR-012 출장 여비 규정


In [11]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

engine=create_engine(f'sqlite:///{db_path}')
SessionLocal=sessionmaker(bind=engine)

with SessionLocal() as session:
    rows=session.execute(
        text('SELECT id, doc_id, title, version, department FROM documents')
    ).all()

    session.commit()

In [12]:
# conn.close()
engine.dispose()